# Triangle Meshes
Triangles are perhaps the simplest of the graphics primitives. Even so, they are used by the millions in modern games. Some are smaller than a single pixel. Here we will learn how to take a mesh of 3D triangles, transform them to camera space, shade them according to their intrinsic color and extrinsic lighting, and draw them.

## In-Memory Meshes
Meshes are traditionally composed of the following:

* A series of vertices
* A series of facets. Each facet consists of a minimum of an ordered tuple of vertex indexes. Each facet might also have some metadata describing how to render the triangle. In our case, we have a flat intrinsic color for each facet

In our case, we read this form, but internally form triangles as two lists:

In [ ]:
from magrathea.doc import display_literate_code
display_literate_code("../src/magrathea/triangle/mesh.py","meshinit")

<img src="triangles_in_memory.svg" width=250>

Each triangle is represented in memory as one slice of a 3D data cube. In that slice, each column represents one vertex and each cell in the column represents the $x$, $y$, or $z$ coordinates respectively. It is represented as an array of size $(M,3,3)$ where the first (slice) index is the triangle number, second (row of slice) is the x,y, or z coordinate, and the third (column of slice) is the corner index. So `self.triangles[37,1,2]` is triangle 37, $y$ component of corner $C$. 

In our current model, we just have flat shading for each triangle. Each triangle has one intrinsic color, and if the light hits it squarely, each pixel in the triangle will be exactly that color. So, we have a second array of size (M,3). Note that we considered (3,M) such that each color is a "vector" and could be linearly transformed, but it turns out that's not terribly useful in our current project. There may be other projects where it is useful (maybe rotation changes hue) but not here. Since our frame buffer is (H,W,3), we use a color map of (M,3) to keep the entry for each triangle broadcast-compatible with the entry for each pixel.

## Loading a mesh
This project's preferred import form is [3D Manufacturing Format](https://en.wikipedia.org/wiki/3D_Manufacturing_Format) with extension `.3mf`. The program [OpenSCAD](openscad.org) (specifically the nightly version) is capable of creating these from `scad` models. So, the workflow is to design the object in OpenSCAD, making sure to color each primitive appropriately. Rendering an object in OpenSCAD breaks it down into a perfect mesh of nonintersecting, consistently-oriented triangles. We'll see why this is important later, but let's just say now that this is "non-trivial".

A 3mf file is really a zip with an xml in it. It's a zip so that it can handle attachments like texture maps, but we aren't using that feature. We only care about 3D/3dmodel.model. We aren't opening a general-purpose 3mf, but just one produced by OpenSCAD. For example, here is a cube in OpenSCAD:

``` scad
color([1,0.5,0])
cube(center=true);
```

And here is the processed 3mf:

``` xml
<?xml version="1.0" encoding="utf-8"?>
<model xmlns="http://schemas.microsoft.com/3dmanufacturing/core/2015/02" unit="meter" xml:lang="en-US" xmlns:m="http://schemas.microsoft.com/3dmanufacturing/material/2015/02" xmlns:p="http://schemas.microsoft.com/3dmanufacturing/production/2015/06" xmlns:b="http://schemas.microsoft.com/3dmanufacturing/beamlattice/2017/02" xmlns:s="http://schemas.microsoft.com/3dmanufacturing/slice/2015/07">
	<metadata name="Title">cube.scad</metadata>
	<metadata name="Application">OpenSCAD (https://www.openscad.org/)</metadata>
	<metadata name="CreationDate">2025-08-30T17:39:45Z</metadata>
	<resources>
		<m:colorgroup id="2">
			<m:color color="#FF8000FF" />
			<m:color color="#F9D72CFF" />
		</m:colorgroup>
		<object id="1" name="OpenSCAD Model" type="model" p:UUID="cbc9b55c-b710-453b-9bba-ea8489113e61" pid="2" pindex="1">
			<mesh>
				<vertices>
					<vertex x="-0.500000" y="-0.500000" z="-0.500000" />
					<vertex x="-0.500000" y="-0.500000" z="0.500000" />
					<vertex x="-0.500000" y="0.500000" z="-0.500000" />
					<vertex x="-0.500000" y="0.500000" z="0.500000" />
					<vertex x="0.500000" y="-0.500000" z="-0.500000" />
					<vertex x="0.500000" y="-0.500000" z="0.500000" />
					<vertex x="0.500000" y="0.500000" z="-0.500000" />
					<vertex x="0.500000" y="0.500000" z="0.500000" />
				</vertices>
				<triangles>
					<triangle v1="0" v2="1" v3="3" pid="2" p1="0" />
					<triangle v1="0" v2="2" v3="6" pid="2" p1="0" />
					<triangle v1="0" v2="3" v3="2" pid="2" p1="0" />
					<triangle v1="0" v2="4" v3="5" pid="2" p1="0" />
					<triangle v1="0" v2="5" v3="1" pid="2" p1="0" />
					<triangle v1="0" v2="6" v3="4" pid="2" p1="0" />
					<triangle v1="1" v2="5" v3="3" pid="2" p1="0" />
					<triangle v1="2" v2="3" v3="6" pid="2" p1="0" />
					<triangle v1="3" v2="5" v3="7" pid="2" p1="0" />
					<triangle v1="3" v2="7" v3="6" pid="2" p1="0" />
					<triangle v1="4" v2="6" v3="5" pid="2" p1="0" />
					<triangle v1="5" v2="6" v3="7" pid="2" p1="0" />
				</triangles>
			</mesh>
		</object>
	</resources>
	<build p:UUID="dddd526d-3c2d-4a52-a684-4caaf058f20d">
		<item objectid="1" p:UUID="1b5e9ed1-f2dd-4ec7-81f2-100d72b52a59" />
	</build>
</model>
```

This contains:
* A color list with two entries
* A list of 8 vertices, one for each corner of the cube
* A list of 12 triangles, 2 for each of the 6 faces of the cube.

Each triangle has three vertex indexes referring to the list of vertices (zero-based) and a color from the list of colors. The vertex indexes are `v1`, `v2`, and `v3`, while the color is entry `p1` from color table id `pid`. We assume there is only one color table so ignore the `pid` and use the first table. Here, all 12 triangles are the same color, and that color is color 0, #FF8000 or orange.

We load the mesh using the following code. This code uses the built-in `zipfile` library to read the zip, and the `xml.etree` library to parse the xml. It builds a list of colors, then a list of vertices, then a list of `triangles` and its corresponding list of `tricolors`. We use fancy indexing to immediately look up the colors in the color table to build the tricolors table, and similarly use fancy indexing to get each vertex out of the vertex list. Finally, it calls the `Mesh()` constructor to create an in-memory mesh.

In [ ]:
display_literate_code("../src/magrathea/triangle/meshload.py","load3mf")

Note that it is a design choice to immediately look up vertices, duplicating them as necessary, to build the triangle mesh in memory. We *could* track vertices and facets internally. This might make it easier to transform, since we only transform the de-duplicated vertex list. I do it this way because our triangle rasterizer will eventually be used to draw each triangle and it works with an (N,2,3) list for $N$ 2D triangles.

In principle we could write similar loaders `loadobj` etc for different formats. As it happens, this one works with our workflow and is sufficient.

## Geometry Shader
For historical reasons, in GPU pipelines, this phase is called the "geometry shader". It is a program run by the same hardware that does other kinds of shading, so it makes a little bit of sense. We implement the geometry shader in the function `Mesh.shade_geometry() which takes its own list of triangles and colors, along with a transformation to universe from body and a transformation to camera from universe, and an offset of the center of the body from the camera.

In [ ]:
display_literate_code("../src/magrathea/triangle/mesh.py","shade_geometry_header")

The geometry shader does the following steps:

### Transformation from body to universe
Transform the vertices from the body frame to a frame centered on the body origin, but parallel to the universe frame. This is a 3x3 matrix linear transformation. Here we take advantage of a design feature of Numpy broadcasting. First, we transform a single vector by treating it as a column vector and multiplying the transformation matrix and the vector. $$\vec{v}_u=\MM{M}{_{ub}}\vec{v}_b$$ Second, we transform a bundle of vectors, say 3 vectors for a triangle like this: We put each column vector into a matrix, transform that matrix, then extract each vector from the result: 

$$\begin{eqnarray}
\MM{V}{_b}&=&\begin{bmatrix}\vec{a}_{b} & \vec{b}_{b} & \vec{c}_{b}\end{bmatrix}\\
 &=&\begin{bmatrix}x_{ab} & x_{bb} & x_{cb}\\
 y_{ab} & y_{bb} & y_{cb}\\
 z_{ab} & z_{bb} & z_{cb}\end{bmatrix}\\
\MM{V}{_u}&=&\MM{M}{_{ub}}\MM{V}{_b}\\
 &=&\begin{bmatrix}x_{au} & x_{bu} & x_{cu}\\
 y_{au} & y_{bu} & y_{cu}\\
 z_{au} & z_{bu} & z_{cu}\end{bmatrix}\\
 &=&\begin{bmatrix}\vec{a}_{u} & \vec{b}_{u} & \vec{c}_{u}\end{bmatrix}
\end{eqnarray}$$ 

These kinds of "broadcasting" are in a sense built into the matrix multiplication of linear algebra, and are not special to Numpy. What *is* however is multiplying one matrix by several matrices. We can multiply a single transformation matrix `M_ub` with shape `(3,3)` by a *bundle* of matrices, one for each triangle, with shape `(M,3,3)`. Each `(...,3,3)` slice is a matrix, and each column of each slice is a vertex. The result is a bundle of transformed matrices. Numpy can do this for any kind of broadcastable bundle of matrices, as long as the last two dimensions of each bundle are the matrix slice, compatible with matrix multiplication, and the first dimensions are compatible by the rules of numpy broadcasting. We therefore do this transformation in a single line:

In [ ]:
display_literate_code("../src/magrathea/triangle/mesh.py","shade_geometry_ub")

### Lambertian Reflectance
Next, we apply the Lambertian shading model. For a long-winded justification, go see [Wikipedia](https://en.wikipedia.org/wiki/Lambertian_reflectance#Use_in_computer_graphics).  One thing to consider is that the light sources are far enough away that they can be considered infinitely far. We are illuminating a 10m scale spacecraft mesh with a light source almost 10^12m away.

#### Normal vectors
To apply the Lambertian reflectance model, we need normal vectors. Here, we are calculating all the normal vectors in universe space. We calculate a vector from corner 0 to corner 1, calling it edge `a`. We calcualte another edge `b` from corner 0 to corner 2. The normal vector is the unit-length cross product of these two vectors. Note that we take advantage of the fact that OpenSCAD carefully constructed our mesh to have triangles of consistent wrapping and orientation, so this normal vector is guaranteed to point "outside". Also note that since we are modeling a spacecraft and not a Klein bottle, it is possible to consistently orient the shape.

In principle we could have transformed straight to camera from body by combining the camera-from-universe matrix with the universe-from-body matrix. This is one simple matrix multiplication. However, I decided to do it in two steps, so that we can calculate shading in the Universe frame. We could in principle calculate the normals in body frame and transform them to camera frame, but we must be careful about this. It turns out that there *is* a linear transform that transforms normal vectors correctly, but it's not the naive transform using the same matrix as for transforming vertices. 

Instead, we work on lighting in the universe frame.



In [ ]:
display_literate_code("../src/magrathea/triangle/mesh.py","shade_geometry_normal")

#### Reflectance model
A reflectance model tells us how bright an object should be, based on its intrinsic brightness, the direction to a light source, and the direction to the camera. So, in general we need
$$\vec{B}=f(\hat{N},\hat{L},\hat{C},\vec{I}_L,\vec{I}_i)$$
where:
* $\vec{B}$ is the perceived brightness. We use a vector because we are treating it as an RGB color.
* $\hat{N}$ is the normal vector at the point
* $\hat{L}$ is the direction (unit vector) from the point to the light source
* $\hat{C}$ is the direction from the point to the camera
* $\vec{I}_L$ is the brightness of the light source, treated as a vector again.
* $\vec{I}_i$ is the intrinsic color of the surface, again an RGB vector.
However the Lambertian reflectance model specifically says that light is scattered equally in all directions, and therefore the direction to the camera is irrelevant to the model. We just need the normal and direction to the camera. We also treat the light source as unit intensity in all components. Further, we use a spectacularly easy-to-calculate function:
$$\begin{eqnarray}
\vec{B}&=&f(\hat{N},\hat{L},\vec{I}_i)\\
 &=&\cos\theta_{NL}\vec{I}_i\\
 &=&(\hat{N}\cdot\hat{L})\vec{I}_i
\end{eqnarray}$$
If this is negative, it means the triangle is facing away from the light and is shaded. It's brightness from this light is therefore clamped to zero. We use Numpy to calculate all the brighnesses at once. We end up with a bundle of scalars for each triangle, or a 1D array.

In [ ]:
display_literate_code("../src/magrathea/triangle/mesh.py","shade_geometry_lambert")

### Transformation to camera space
Now that the brightness model has been run, we have finished with universe space and can transform into camera space. This is the camera-from-universe matrix, *plus* the offset between the center of the body and the camera in camera space. Otherwise, the camera would be at the center of the object and try to render it from the inside.

Camera space, as discussed elsewhere, is a normalized space which will be easy to convert to pixels, but is independent of pixels. It has the following basis vectors, in order:
* A vector parallel to the horizontal direction of the final image, with positive on the *right*. +0.5 is on the right edge of the image and -0.5 is on the left edge.
* A vector parallel to the vertical direction of the final image, with positive *downward*. +0.5 is on the bottom edge of the image, and -0.5 is on the top edge.
* A vector parallel to the boresight, facing the direction the camera is looking. The camera is at 0.0, and the virtual image plane is at +1.0.

We can control the aspect ratio of the image with the relative length of the $\hat{x}_c$ and $\hat{y}_c$ basis vectors in universe space, and we can control the focal length with the length of the $\hat{z}_c$ basis vector.

Matrix multiplication brings this into a frame parallel to camera space but *still* centered at the origin of the body. We fix this by deciding where the mesh should appear relative to the camera, and adding this offset to each vertex. Once again broadcasting works for us. The offset `T_c` is a `(3,1)` column vector, added to a `(M,3,3)` bundle of triangles. `T_c` is widened to `(1,3,1)` and then expanded to `(M,3,3)` so it can be added element-by-element. This is all done with pointer magic so that no extra memory is needed for these extra elements.

In [ ]:
display_literate_code("../src/magrathea/triangle/mesh.py","shade_geometry_Mcu")

### Triangle Culling
Some triangles are on the back of the object. While we *could* draw such triangles, they will *always* be completely obscured by triangles that are facing us. Therefore as an optimization, we will *cull* triangles that are facing away from us. While we are at it, we will also cull triangles that are behind us ($z_c<0$). We recalculate the normal vectors in camera space, because this is easier to read than calculating the transformation of normals from one frame to another.

We calculate the edges as before, then calculate the cross product, then calculate the dot product between the direction to one corner. Normally we would be careful with magnitudes and normalization, but not this time. The away or facing decision is based solely on the dot product being greater or less than zero, and this depends solely on the angle being less than or greater than 90deg respectively.

We end up with a `keep_mask` -- this was called `cull_mask` in the past but was always `True` when we keep a triangle and `False` otherwise. In either case this is a 1D array of shape `(M,)` elements, of which `(N,)` Then we use Numpy fancy addressing to collect all the triangles we will keep into one array, and discard the ones we won't.

#### Fancy Addressing
Numpy arrays can be indexed with boolean masks. A 1D mask will be applied to the first array, so for instance if you have a bundle of triangles of shape `(M,3,3)` and a mask of shape `(M,)` of which `N` of the values are true, those values will be selected. The expression will have shape `(N,3,3)`. The mask is applied to the dimension on the left. If the mask is 2D, for instance `in_triangle` with shape `(H,W)` of which some `N` are `True` and the rest are `False`, you can apply this to an RGB image of shape `(H,W,3)`. All of the dimensions of the mask have to match the sizes of the corresponding leftmost dimensions of the masked array. The result is of shape `(N,3)`. Furthermore, this is a *view*, meaning it shares its values with the original array. If you write to this view, it will affect the original array. This allows things like `frame_buffer[in_triangle]=tricolor`. The view `frame_buffer[in_triangle]` on the left side has shape `(N,3)`, which broadcasts to the `tricolor` shape `(3,)`. All selected pixels in `frame_buffer` are set to the chosen value.

With this, we do culling the Numpythonic way:

In [ ]:
display_literate_code("../src/magrathea/triangle/mesh.py","shade_geometry_cull")

The code culls the triangles, leaving the kept triangles in `tris_kept`. Further, it culls the `tricolors` and Lambertian brightnesses. It computes the final shade of each triangle, determined by scaling the Lambertian brightness, and adding a component of ambient brightness to keep the shaded side from being completely dark. Then it multiplies the kept intrinsics by the shades so we have a final solid color for each triangle in `trishades_kept`. The result is a bundle of triangles `tris_kept` of shape `(N,3,3)` and a bundle of shades `trishades_kept` of shape `(N,3)`.

### Z sorting
The Painter's Algorithm is a method of hidden surface removal. We draw the triangles (and all primitives) on the frame buffer in order from furthest from camera to closest. If a closer primitive happens to occupy the same pixel as the further primitive, the further one is erased and replaced with the closer one.

To apply this to a mesh, we sort the triangles in order from largest distance to camera to smallest. We actually compare the *squared* distance calculated by vdot of corner 0 with itself. This is valid since `sqrt()` is a monotonic function. Further, we sort by the *negative* of the squared distance, so that we don't need to do anything special with our sorting routine.

#### Permutation arrays
Finally, we use the `np.argsort()` routine. This sorts an array, but doesn't return a sorted array. Instead it results in a permutation array. For instance, suppose we have the following array:

In [ ]:
import numpy as np
unsorted=np.array([18,2,9,0,1])
names=np.array(["Alice","Bob","Carl","Doug","Eddie"])

We use `np.argsort` to determine what is needed to sort the unsorted array. We then can apply this to both `unsorted` and `names`:

In [ ]:
s=np.argsort(unsorted)
print(f"{s=}")
print(f"{unsorted[s]=}")
print(f"{names[s]=}")

We use this technique to get the permutation from the distances, then apply it to the triangles and the tricolors:

In [ ]:
display_literate_code("../src/magrathea/triangle/mesh.py","shade_geometry_sort")

### Projection
Here comes the sole non-linear step in this process. We need to project the vectors. If a vertex is a given distance off axis but twice as far away along the Z-axis, it appears to be half the distance from the center. It's non-linear, meaning we can't build a matrix that does this, but it's still simple. All we have to do is divide all of the coordinates by the $z$ coordinate. This has the effect of sqhishing everything to the plane $z_c=1.0$.

Further, if the size of the frame buffer is passed in, we can convert from $[-0.5,0.5]$ to $[0,w)$ or $[0,h)$. This gives us the opportunity to discuss center of pixels. Mostly this is just the process of figuring out how our mapping...

In [ ]:
display_literate_code("../src/magrathea/triangle/mesh.py","shade_geometry_project")

## Triangle Rasterization


## Putting it all together
First we load a mesh. For quicker rendering, use the cube. For closer to our intent, use the Voyager spacecraft. Since the cube is closer, we will use the smaller $z_c$. We will also recolor all the triangles of the cube internally so that we can see how the square faces are split.

In [ ]:
from kwanmath.matrix import rot_x
from magrathea.triangle.tridraw_cy import py_tris_draw
from matplotlib import pyplot as plt

from magrathea.triangle.meshload import load3mf
from magrathea.triangle.tridraw import tri_raster

use_cube=False
if use_cube:
    mesh=load3mf("../data/output/mesh/cube.3mf")
    z_c=3
    resistor_color_code=np.array([[0.25,0.25,0.25],
                                  [0.5,0.2,0.0],
                                  [1.0,0.0,0.0],
                                  [1.0,0.5,0.0],
                                  [1.0,1.0,0.0],
                                  [0.0,1.0,0.0],
                                  [0.0,0.0,1.0],
                                  [0.5,0.0,1.0],
                                  [0.5,0.5,0.5],
                                  [1.0,1.0,1.0]])
    mesh.tricolors = resistor_color_code[
        np.arange(mesh.tricolors.shape[0], dtype=np.int32) % resistor_color_code.shape[0], :]
else:
    mesh=load3mf("../data/output/mesh/voyager.3mf")
    z_c=20
    

Now we render the mesh 63 times, rotating it 0.1 radian each time. We generate a universe-from-body matrix that rotates the body around the X axis. We then run the geometry shader to get shaded 2D triangles, and the triangle rasterizer to get the triangles into the frame buffer.

In [ ]:
verbose=True
if verbose:
    plt.figure()
print(mesh)
thetas=np.arange(63)/10
for i_theta,theta in enumerate(thetas):
    M_ub=rot_x(theta)
    frame_buffer = np.zeros((500, 500, 3))
    tris_screen, tricolors_screen = mesh.shade_geometry(M_ub=M_ub,
                                                        M_cu=np.eye(3),
                                                        T_c=np.array([[0], [0], [z_c]]),
                                                        lhat_u=np.array([[0], [0], [-1]]), diffuse=0.9, ambient=0.1,
                                                        w=frame_buffer.shape[1], h=frame_buffer.shape[0])
    #py_tris_draw(frame_buffer,tris_screen,tricolors_screen)
    for tri, tricolor in zip(tris_screen,tricolors_screen):
        tri_raster(frame_buffer, tricolor, tri[0,0], tri[1,0], tri[0,1], tri[1,1], tri[0,2], tri[1,2])
    if verbose:
        plt.clf()
        plt.imshow(frame_buffer)
        plt.title(f"{i_theta}: {theta}")
        plt.pause(0.1)
    #break
if verbose:
    plt.show()